# CENG 467 — Week 1 Runbook (Colab)

Day-by-day execution. **Run cells top-to-bottom**, verify the output of each day before moving on. Total cost ~$5.50, total wall-clock ~6-8 hr.


## Day 1 — Setup


In [ ]:
# Mount Drive (so caches survive Colab session timeouts)
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib
WORK = pathlib.Path('/content/drive/MyDrive/ceng467_termproject')
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)


In [ ]:
# Clone the repo (replace URL with your fork)
REPO_URL = 'https://github.com/cagancaliskan/turkish-summarization-distillation.git'
if not pathlib.Path('turkish-summarization-distillation').exists():
    !git clone $REPO_URL
os.chdir('turkish-summarization-distillation')
!ls


In [ ]:
# Install dependencies — order matters on Colab.
# 1. Bring our pinned packages in WITHOUT downgrading torch/torchvision/torchaudio
#    (Colab ships torch 2.10 + CUDA 12.8; downgrading breaks torchvision::nms).
# 2. Bring our pinned packages in WITHOUT downgrading protobuf below TF's minimum
#    (Colab's TF 2.20 needs protobuf >= 5.28).
!pip install -q -r requirements.txt
# Final verification — import everything we need.
!python -c "import torch, transformers, peft, openai, anthropic, datasets, rouge_score, bert_score; print('OK', 'torch', torch.__version__, 'transformers', transformers.__version__, 'cuda', torch.cuda.is_available())"


In [ ]:
# API keys — paste into the os.environ statements below.
import os
os.environ['OPENAI_API_KEY']    = 'sk-...REPLACE_ME...'
os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...REPLACE_ME...'
assert os.environ['OPENAI_API_KEY'].startswith('sk-')
assert os.environ['ANTHROPIC_API_KEY'].startswith('sk-ant-')
print('keys set')


## Day 2 — Datasets + 100-article pilot
Should take ~30-45 min for the first download. **Verify counts match before continuing.**


In [ ]:
!python -m src.data.load_mlsum --out-dir data/raw/mlsum_tr


In [ ]:
!python -m src.data.load_trnews --out data/raw/trnews/test.jsonl --n 1000


In [ ]:
!python -m src.data.make_pilot \
    --input data/raw/mlsum_tr/train.jsonl \
    --out   data/raw/mlsum_tr/pilot_100.jsonl \
    --n 100


In [ ]:
# Verify
!wc -l data/raw/mlsum_tr/*.jsonl data/raw/trnews/*.jsonl 2>/dev/null
!head -n 1 data/raw/mlsum_tr/pilot_100.jsonl | python -m json.tool | head -n 14


## Day 3 — Validate prompts on 50 articles each (~$0.05)
Stop after this and review the printed samples. If anything looks wrong, ping me before Day 4.


In [ ]:
PILOT='data/raw/mlsum_tr/pilot_100.jsonl'
!python -m src.teachers.generate --teacher openai    --prompt-variant concise  --input $PILOT --n 50 --out-dir data/synthetic/openai/concise
!python -m src.teachers.generate --teacher openai    --prompt-variant detailed --input $PILOT --n 50 --out-dir data/synthetic/openai/detailed
!python -m src.teachers.generate --teacher anthropic --prompt-variant concise  --input $PILOT --n 50 --out-dir data/synthetic/anthropic/concise
!python -m src.teachers.generate --teacher anthropic --prompt-variant detailed --input $PILOT --n 50 --out-dir data/synthetic/anthropic/detailed


In [ ]:
!python -m scripts.inspect_outputs --pilot data/raw/mlsum_tr/pilot_100.jsonl \
    --concise  data/synthetic/openai/concise \
    --detailed data/synthetic/openai/detailed --k 5


In [ ]:
!python -m scripts.inspect_outputs --pilot data/raw/mlsum_tr/pilot_100.jsonl \
    --concise  data/synthetic/anthropic/concise \
    --detailed data/synthetic/anthropic/detailed --k 5


## Day 4 — Full 10k GPT-4o-mini concise (~2.5 hr, ~$1.50)
Resumable; if Colab disconnects, just re-run the cell.


In [ ]:
!bash scripts/02_generate_teacher.sh openai concise 10000


In [ ]:
!python -m scripts.check_cache --input data/raw/mlsum_tr/train.jsonl \
    --cache-dir data/synthetic/openai/concise --n 10000


## Day 5 — Full 10k Claude 3 Haiku concise (~3 hr, ~$2.50)


In [ ]:
!bash scripts/02_generate_teacher.sh anthropic concise 10000


In [ ]:
!python -m scripts.check_cache --input data/raw/mlsum_tr/train.jsonl \
    --cache-dir data/synthetic/anthropic/concise --n 10000


## Day 6 — 1k detailed-prompt subsets per teacher (~1.5 hr, ~$0.40)


In [ ]:
!bash scripts/02_generate_teacher.sh openai    detailed 1000
!bash scripts/02_generate_teacher.sh anthropic detailed 1000


In [ ]:
!python -m scripts.check_cache --input data/raw/mlsum_tr/train.jsonl --cache-dir data/synthetic/openai/detailed    --n 1000
!python -m scripts.check_cache --input data/raw/mlsum_tr/train.jsonl --cache-dir data/synthetic/anthropic/detailed --n 1000


## Day 7 — Zero-shot baselines on the MLSUM test set (~1.5 hr GPU/API, ~$1.30)


In [ ]:
TEST='data/raw/mlsum_tr/test.jsonl'
!mkdir -p outputs/predictions
# B1 — zero-shot mT5-small (downloads ~1.2 GB once)
!python -m src.student.infer --model-path google/mt5-small --input $TEST --out outputs/predictions/B1_zeroshot.jsonl


In [ ]:
# B3a — GPT-4o-mini zero-shot on test set
!python -m src.student.infer_teacher --teacher openai --prompt-variant concise \
    --input $TEST --out outputs/predictions/B3a_gpt.jsonl


In [ ]:
# B3b — Claude 3 Haiku zero-shot on test set
!python -m src.student.infer_teacher --teacher anthropic --prompt-variant concise \
    --input $TEST --out outputs/predictions/B3b_claude.jsonl


In [ ]:
# Sanity metrics (ROUGE + error flags only — fast)
!python -m src.eval.run_eval \
    --pred B1=outputs/predictions/B1_zeroshot.jsonl \
    --pred B3a=outputs/predictions/B3a_gpt.jsonl \
    --pred B3b=outputs/predictions/B3b_claude.jsonl \
    --metrics rouge errors \
    --out-json outputs/results/week1_baselines.json
import json, pandas as pd
with open('outputs/results/week1_baselines.json') as f: d = json.load(f)
rows = [{'sys': k, 'n': v['n'], **{f'r1_{m}': v[f'rouge_{m}']['rouge1'] for m in ('standard','stem')}, 'frac_halluc#': v['errors']['frac_hallucinated_numbers']} for k,v in d.items()]
pd.DataFrame(rows).round(4)


## Week 1 done

Tick all seven items in `WEEK1_RUNBOOK.md` and ping me. I will not move on to Week 2 until you confirm.
